**Data note.** The original data file (`world_alcohol.csv`) is not in this repository; this notebook now downloads the World Health Organization Global Health Observatory table of recorded alcohol consumption per person (aged 15+, litres of pure alcohol) by beverage type, 1960 to 2024, from the GHO OData API (https://ghoapi.azureedge.net/api/SA_0000001400), WHO data terms CC BY 4.0, instead; results differ from the earlier outputs. The original file held 100 rows for 1985 to 1989; the downloaded table has about 40,000 rows, so counts and filtered rows differ. The column names are unchanged. `random_state=0` was added to the two `df.sample` calls and a typo (`w_a_con` for `df`) was fixed.

In [1]:
# Data: WHO Global Health Observatory (GHO), indicator SA_0000001400
# "Alcohol, recorded per capita (15+) consumption (in litres of pure alcohol), by beverage type".
# The table is rebuilt with the same five columns the rest of this notebook uses.
import json, os, urllib.request
import pandas as pd

os.makedirs("data", exist_ok=True)
WA_PATH = "data/world_alcohol.csv"
GHO = "https://ghoapi.azureedge.net/api/"


def gho(path):
    with urllib.request.urlopen(GHO + path, timeout=180) as r:
        return json.load(r)["value"]


if not os.path.exists(WA_PATH):
    raw = pd.DataFrame(gho("SA_0000001400"))
    country_names = {v["Code"]: v["Title"] for v in gho("DIMENSION/COUNTRY/DimensionValues")}
    beverages = {
        "ALCOHOLTYPE_SA_WINE": "Wine",
        "ALCOHOLTYPE_SA_BEER": "Beer",
        "ALCOHOLTYPE_SA_SPIRITS": "Spirits",
        "ALCOHOLTYPE_SA_OTHER_ALCOHOL": "Other",
    }
    raw = raw[raw["Dim1"].isin(beverages)]  # the "total" rows are left out
    pd.DataFrame({
        "Year": raw["TimeDim"],
        "WHO region": raw["ParentLocation"],
        "Country": raw["SpatialDim"].map(country_names),
        "Beverage Types": raw["Dim1"].map(beverages),
        "Display Value": raw["NumericValue"].round(2),
    }).to_csv(WA_PATH, index=False)
print("Rows in", WA_PATH, "=", len(pd.read_csv(WA_PATH)))

Rows in data/world_alcohol.csv = 41521


In [2]:
import pandas as pd
# World alcohol consumption data
df = pd.read_csv(WA_PATH)
print("World alcohol consumption sample data:")
print(df.head())
print('\nShape of the dataframe: ',df.shape)
print('\nNumber of rows: ',df.shape[0])
print('\nNumber of column: ',df.shape[1])
print("\nExtract Column Names:")
print(df.columns)

World alcohol consumption sample data:
   Year WHO region                             Country Beverage Types  \
0  2023     Africa                        South Africa           Wine   
1  2024     Africa                        South Africa           Wine   
2  2023   Americas  Venezuela (Bolivarian Republic of)           Wine   
3  2024   Americas  Venezuela (Bolivarian Republic of)           Wine   
4  2023     Africa                             Liberia           Wine   

   Display Value  
0           1.33  
1           1.27  
2           0.04  
3           0.04  
4           0.01  

Shape of the dataframe:  (41521, 5)

Number of rows:  41521

Number of column:  5

Extract Column Names:
Index(['Year', 'WHO region', 'Country', 'Beverage Types', 'Display Value'], dtype='object')


In [3]:

print(df.head())
print("\nSelect first 2 rows:")
print(df.iloc[:2])
print("\nSelect first 2 columns:")
print(df.iloc[:,:2].head())
print("\nSelect 2 specific columns:")
print(df[['Display Value', 'Year']])

   Year WHO region                             Country Beverage Types  \
0  2023     Africa                        South Africa           Wine   
1  2024     Africa                        South Africa           Wine   
2  2023   Americas  Venezuela (Bolivarian Republic of)           Wine   
3  2024   Americas  Venezuela (Bolivarian Republic of)           Wine   
4  2023     Africa                             Liberia           Wine   

   Display Value  
0           1.33  
1           1.27  
2           0.04  
3           0.04  
4           0.01  

Select first 2 rows:
   Year WHO region       Country Beverage Types  Display Value
0  2023     Africa  South Africa           Wine           1.33
1  2024     Africa  South Africa           Wine           1.27

Select first 2 columns:
   Year WHO region
0  2023     Africa
1  2024     Africa
2  2023   Americas
3  2024   Americas
4  2023     Africa

Select 2 specific columns:
       Display Value  Year
0               1.33  2023
1              

In [4]:
import pandas as pd
# World alcohol consumption data
df = pd.read_csv(WA_PATH)
print("World alcohol consumption sample data:")
print(df.head())
print("\nSelect random number of rows:")
print(df.sample(5, random_state=0))
print("\nSelect fraction of randome rows:")
print(df.sample(frac=0.2, random_state=0))

World alcohol consumption sample data:
   Year WHO region                             Country Beverage Types  \
0  2023     Africa                        South Africa           Wine   
1  2024     Africa                        South Africa           Wine   
2  2023   Americas  Venezuela (Bolivarian Republic of)           Wine   
3  2024   Americas  Venezuela (Bolivarian Republic of)           Wine   
4  2023     Africa                             Liberia           Wine   

   Display Value  
0           1.33  
1           1.27  
2           0.04  
3           0.04  
4           0.01  

Select random number of rows:
       Year             WHO region                           Country  \
21288  1986  Eastern Mediterranean              Syrian Arab Republic   
34102  1989  Eastern Mediterranean                             Egypt   
28149  1968        Western Pacific  Lao People's Democratic Republic   
18475  2004  Eastern Mediterranean              United Arab Emirates   
29851  2001      

In [5]:
print("\nMissing values:")
print(df.isnull())
print("\nDropping the missing values:")
print(df.dropna())


Missing values:
        Year  WHO region  Country  Beverage Types  Display Value
0      False       False    False           False          False
1      False       False    False           False          False
2      False       False    False           False          False
3      False       False    False           False          False
4      False       False    False           False          False
...      ...         ...      ...             ...            ...
41516  False       False    False           False          False
41517  False       False    False           False          False
41518  False       False    False           False          False
41519  False       False    False           False          False
41520  False       False    False           False          False

[41521 rows x 5 columns]

Dropping the missing values:
       Year       WHO region                                Country  \
0      2023           Africa                           South Africa   
1    

In [6]:
print(df.drop_duplicates('WHO region'))

    Year             WHO region                             Country  \
0   2023                 Africa                        South Africa   
2   2023               Americas  Venezuela (Bolivarian Republic of)   
6   2023                 Europe                              Serbia   
12  2023        Western Pacific                            Cambodia   
14  2023  Eastern Mediterranean                United Arab Emirates   
58  2023        South-East Asia                              Bhutan   

   Beverage Types  Display Value  
0            Wine           1.33  
2            Wine           0.04  
6            Wine           2.71  
12           Wine           0.02  
14           Wine           0.40  
58           Wine           0.05  


In [7]:
print(df.head())
print("\nThe world alcohol consumption details in the year 1985:")
print(df[df['Year']==1985].head(10))
print("\nThe world alcohol consumption details in the year 1989:")
print(df[df['Year']==1989].head(10))

   Year WHO region                             Country Beverage Types  \
0  2023     Africa                        South Africa           Wine   
1  2024     Africa                        South Africa           Wine   
2  2023   Americas  Venezuela (Bolivarian Republic of)           Wine   
3  2024   Americas  Venezuela (Bolivarian Republic of)           Wine   
4  2023     Africa                             Liberia           Wine   

   Display Value  
0           1.33  
1           1.27  
2           0.04  
3           0.04  
4           0.01  

The world alcohol consumption details in the year 1985:
      Year             WHO region                           Country  \
1712  1985                 Africa  Democratic Republic of the Congo   
1741  1985               Americas               Antigua and Barbuda   
1745  1985                 Africa                           Namibia   
1929  1985               Americas                         Argentina   
1972  1985               Americas  

In [8]:
print("\nThe world alcohol consumption details where year is 1987 or 1989:")
print((df[(df['Year']==1987) | (df['Year']==1989)]).head(10))


The world alcohol consumption details where year is 1987 or 1989:
      Year             WHO region  \
1707  1989        South-East Asia   
1708  1987  Eastern Mediterranean   
1714  1989                 Africa   
1716  1989                 Africa   
1790  1987        Western Pacific   
1813  1987  Eastern Mediterranean   
1824  1987                 Europe   
1862  1989                 Europe   
1869  1989                 Africa   
1914  1989                 Europe   

                                                Country Beverage Types  \
1707                                             Bhutan           Beer   
1708                                               Iraq           Wine   
1714                                            Eritrea          Other   
1716                                            Comoros           Wine   
1790                                          Singapore          Other   
1813                                              Qatar           Wine   
1824  U

In [9]:
print("\nThe world alcohol consumption details by the Americas in the 1985:")
print(df[(df['Year']==1989) & (df['WHO region']=='Americas')])


The world alcohol consumption details by the Americas in the 1985:
       Year WHO region                   Country Beverage Types  Display Value
2043   1989   Americas                    Guyana          Other           0.00
2480   1989   Americas     Saint Kitts and Nevis           Beer           2.27
2829   1989   Americas                   Bahamas        Spirits           7.45
3339   1989   Americas                   Uruguay           Wine           3.72
3506   1989   Americas  United States of America           Beer           4.96
...     ...        ...                       ...            ...            ...
40069  1989   Americas                    Canada           Beer           4.82
40606  1989   Americas                  Paraguay           Beer           2.03
41017  1989   Americas               Saint Lucia           Beer           2.36
41061  1989   Americas                    Guyana        Spirits           8.90
41123  1989   Americas                    Mexico        Spirits

In [10]:
print("\nThe world alcohol consumption details in the year 1986 where WHO region is Western Pacific and country is VietNam :")
print(df[(df['Year']==1986) & (df['WHO region']=='Western Pacific') & (df['Country']=='Viet Nam')])



The world alcohol consumption details in the year 1986 where WHO region is Western Pacific and country is VietNam :
       Year       WHO region   Country Beverage Types  Display Value
14066  1986  Western Pacific  Viet Nam        Spirits           0.05
29087  1986  Western Pacific  Viet Nam           Wine           0.00
37249  1986  Western Pacific  Viet Nam           Beer           0.12


In [11]:
print("\nThe world alcohol consumption details in the year '1986' or '1989' where  WHO region is 'Americas' :")
print(df[((df['Year']==1986) | (df['Year']==1989))  & (df['WHO region']=='Americas')])



The world alcohol consumption details in the year '1986' or '1989' where  WHO region is 'Americas' :
       Year WHO region                Country Beverage Types  Display Value
1891   1986   Americas                  Haiti           Beer           0.00
2043   1989   Americas                 Guyana          Other           0.00
2056   1986   Americas               Suriname           Beer           2.14
2480   1989   Americas  Saint Kitts and Nevis           Beer           2.27
2743   1986   Americas     Dominican Republic          Other           0.00
...     ...        ...                    ...            ...            ...
41041  1986   Americas                Jamaica        Spirits           1.09
41042  1986   Americas               Suriname           Wine           0.09
41061  1989   Americas                 Guyana        Spirits           8.90
41123  1989   Americas                 Mexico        Spirits           1.48
41161  1986   Americas                 Guyana          Other  

In [12]:
print("\nThe world alcohol consumption details in the year ‘1986’ or ‘1989’ where  WHO region is ‘Americas’  or 'Europe':")
print(df[((df['Year']==1985) | (df['Year']==1989)) & ((df['WHO region']=='Americas') | (df['WHO region']=='Europe'))])


The world alcohol consumption details in the year ‘1986’ or ‘1989’ where  WHO region is ‘Americas’  or 'Europe':
       Year WHO region                       Country Beverage Types  \
1741   1985   Americas           Antigua and Barbuda          Other   
1862   1989     Europe                       Iceland           Beer   
1914   1989     Europe                       Denmark        Spirits   
1929   1985   Americas                     Argentina           Wine   
1955   1989     Europe                      Bulgaria        Spirits   
...     ...        ...                           ...            ...   
41044  1989     Europe            Russian Federation          Other   
41061  1989   Americas                        Guyana        Spirits   
41123  1989   Americas                        Mexico        Spirits   
41289  1985     Europe                       Germany           Beer   
41366  1989     Europe  Netherlands (Kingdom of the)        Spirits   

       Display Value  
1741      

In [13]:
print("\nThe world alcohol consumption details ('WHO region','Country','Beverage Types') \nin the year ‘1986’ or ‘1989’ where  WHO region is ‘Americas’  or 'Europe':")
print(df[((df['Year']==1985) | (df['Year']==1989)) & ((df['WHO region']=='Americas') | (df['WHO region']=='Europe'))][['WHO region','Country','Beverage Types']].head(10))



The world alcohol consumption details ('WHO region','Country','Beverage Types') 
in the year ‘1986’ or ‘1989’ where  WHO region is ‘Americas’  or 'Europe':
     WHO region              Country Beverage Types
1741   Americas  Antigua and Barbuda          Other
1862     Europe              Iceland           Beer
1914     Europe              Denmark        Spirits
1929   Americas            Argentina           Wine
1955     Europe             Bulgaria        Spirits
1972   Americas           Costa Rica        Spirits
1987     Europe              Albania           Beer
2043   Americas               Guyana          Other
2066     Europe              Austria           Beer
2402   Americas             Paraguay           Beer


In [14]:
print("\nThe world alcohol consumption details: average consumption of \nbeverages per person >=5 and Beverage Types is Beer:")
print(df[(df['Display Value'] >= 5) & (df['Beverage Types'] == 'Beer')].head(10))


The world alcohol consumption details: average consumption of 
beverages per person >=5 and Beverage Types is Beer:


      Year WHO region  Country Beverage Types  Display Value
1302  2023   Americas   Brazil           Beer           5.22
1303  2024   Americas   Brazil           Beer           5.21
1328  2023     Europe  Austria           Beer           6.18
1329  2024     Europe  Austria           Beer           6.06
1332  2023     Africa    Gabon           Beer           5.98
1333  2024     Africa    Gabon           Beer           6.27
1362  2023     Europe  Croatia           Beer           5.63
1363  2024     Europe  Croatia           Beer           5.70
1394  2023   Americas   Panama           Beer           5.50
1395  2024   Americas   Panama           Beer           5.71


In [15]:
print("\nFiltering records by label or index:")
print(df.loc[0:4, ["WHO region", "Beverage Types"]])


Filtering records by label or index:


  WHO region Beverage Types
0     Africa           Wine
1     Africa           Wine
2   Americas           Wine
3   Americas           Wine
4     Africa           Wine


In [16]:
print("\nThe world alcohol consumption details: average consumption of \nbeverages per person >=4 and Beverage Types is Beer:")
print(df[(df['Display Value'] >= 4) & ((df['Beverage Types'] == 'Beer') | (df['Beverage Types'] == 'Wine')| (df['Beverage Types'] == 'Spirits'))].head(10))



The world alcohol consumption details: average consumption of 
beverages per person >=4 and Beverage Types is Beer:
     Year       WHO region     Country Beverage Types  Display Value
18   2023  Western Pacific   Australia           Wine           4.24
19   2024  Western Pacific   Australia           Wine           4.09
27   2024           Europe    Slovenia           Wine           4.60
114  2023           Europe    Portugal           Wine           7.08
115  2024           Europe    Portugal           Wine           6.81
142  2023           Europe  Luxembourg           Wine           4.52
143  2024           Europe  Luxembourg           Wine           4.41
180  2024           Europe       Spain           Wine           4.06
189  2023           Europe      France           Wine           5.42
190  2024           Europe      France           Wine           5.10


In [17]:
ndf = df.dropna()
print("\nMatch if  a given column has a particular sub string:")
print(ndf[ndf["WHO region"].str.contains("Ea")])


Match if  a given column has a particular sub string:
       Year             WHO region                                Country  \
14     2023  Eastern Mediterranean                   United Arab Emirates   
15     2024  Eastern Mediterranean                   United Arab Emirates   
16     2023  Eastern Mediterranean                                  Libya   
17     2024  Eastern Mediterranean                                  Libya   
40     2023  Eastern Mediterranean                                   Oman   
...     ...                    ...                                    ...   
41464  1974        South-East Asia                            Timor-Leste   
41467  1962  Eastern Mediterranean                               Djibouti   
41483  1987  Eastern Mediterranean                                Somalia   
41514  1987        South-East Asia                                  India   
41516  1975        South-East Asia  Democratic People's Republic of Korea   

      Beverage Types

In [18]:
flt_wine = ndf["WHO region"].isin(["Africa", "Eastern Mediterranean", "Europe"])
print(ndf[flt_wine])

       Year WHO region       Country Beverage Types  Display Value
0      2023     Africa  South Africa           Wine           1.33
1      2024     Africa  South Africa           Wine           1.27
4      2023     Africa       Liberia           Wine           0.01
5      2024     Africa       Liberia           Wine           0.01
6      2023     Europe        Serbia           Wine           2.71
...     ...        ...           ...            ...            ...
41507  2021     Africa          Chad        Spirits           0.00
41509  1970     Africa          Mali           Beer           0.03
41511  2004     Europe        Sweden           Wine           2.68
41517  2017     Europe        Serbia        Spirits           2.85
41518  2017     Europe      Slovakia           Beer           3.43

[25802 rows x 5 columns]

In [19]:
who_region = ["Africa", "Eastern Mediterranean", "Europe"]
flt_wine = ~ndf["WHO region"].isin(who_region)
print(ndf[flt_wine])

       Year       WHO region                                Country  \
2      2023         Americas     Venezuela (Bolivarian Republic of)   
3      2024         Americas     Venezuela (Bolivarian Republic of)   
12     2023  Western Pacific                               Cambodia   
13     2024  Western Pacific                               Cambodia   
18     2023  Western Pacific                              Australia   
...     ...              ...                                    ...   
41514  1987  South-East Asia                                  India   
41515  1998  Western Pacific                      Republic of Korea   
41516  1975  South-East Asia  Democratic People's Republic of Korea   
41519  1976         Americas       Bolivia (Plurinational State of)   
41520  1998         Americas                               Suriname   

      Beverage Types  Display Value  
2               Wine           0.04  
3               Wine           0.04  
12              Wine           0.

In [20]:
print("\nFilter all records where the average consumption of beverages per person from .5 to 2.50.:")
print(df[(df['Display Value'] < 2.5) & (df['Display Value']>.5)])


Filter all records where the average consumption of beverages per person from .5 to 2.50.:
       Year       WHO region                           Country Beverage Types  \
0      2023           Africa                      South Africa           Wine   
1      2024           Africa                      South Africa           Wine   
10     2023           Europe                            Norway           Wine   
11     2024           Europe                            Norway           Wine   
20     2023           Europe                            Cyprus           Wine   
...     ...              ...                               ...            ...   
41502  1962           Africa                         Mauritius        Spirits   
41508  2002         Americas                          Barbados           Wine   
41513  2014  Western Pacific                          Cambodia           Beer   
41515  1998  Western Pacific                 Republic of Korea           Beer   
41519  1976      

In [21]:
print("\nAverage consumption of wine per person greater than 2:")
print(df[(df['Beverage Types'] == 'Wine') & (df['Display Value'] > .2)].count())


Average consumption of wine per person greater than 2:
Year              4745
WHO region        4745
Country           4745
Beverage Types    4745
Display Value     4745
dtype: int64


In [22]:
print("\nFilter rows based on row numbers ended with 0, like 0, 10, 20, 30:")
print(df.filter(regex='0$', axis=0))


Filter rows based on row numbers ended with 0, like 0, 10, 20, 30:


       Year             WHO region              Country Beverage Types  \
0      2023                 Africa         South Africa           Wine   
10     2023                 Europe               Norway           Wine   
20     2023                 Europe               Cyprus           Wine   
30     2023               Americas               Brazil           Wine   
40     2023  Eastern Mediterranean                 Oman           Wine   
...     ...                    ...                  ...            ...   
41480  1996        Western Pacific                Samoa          Other   
41490  1971               Americas  Antigua and Barbuda           Wine   
41500  1968               Americas             Honduras           Wine   
41510  1963        Western Pacific     Papua New Guinea           Beer   
41520  1998               Americas             Suriname        Spirits   

       Display Value  
0               1.33  
10              2.41  
20              0.88  
30              0.2

In [23]:
print(df.loc[:,"Country":"Display Value"].head())

                              Country Beverage Types  Display Value
0                        South Africa           Wine           1.33
1                        South Africa           Wine           1.27
2  Venezuela (Bolivarian Republic of)           Wine           0.04
3  Venezuela (Bolivarian Republic of)           Wine           0.04
4                             Liberia           Wine           0.01


In [24]:
print(df.iloc[:,2:5].head())

                              Country Beverage Types  Display Value
0                        South Africa           Wine           1.33
1                        South Africa           Wine           1.27
2  Venezuela (Bolivarian Republic of)           Wine           0.04
3  Venezuela (Bolivarian Republic of)           Wine           0.04
4                             Liberia           Wine           0.01


In [25]:
print("\nSelect rows with Index label 0 to 9 with specific columns:")
print(df.loc[0:9,["Year","Country","Display Value"]])


Select rows with Index label 0 to 9 with specific columns:
   Year                             Country  Display Value
0  2023                        South Africa           1.33
1  2024                        South Africa           1.27
2  2023  Venezuela (Bolivarian Republic of)           0.04
3  2024  Venezuela (Bolivarian Republic of)           0.04
4  2023                             Liberia           0.01
5  2024                             Liberia           0.01
6  2023                              Serbia           2.71
7  2024                              Serbia           2.75
8  2023                             Germany           3.01
9  2024                             Germany           3.03


In [26]:
print("\nRenaming only some of the column names:")
df.rename(columns = {"WHO region":"WHO_region","Display Value":"Display_Value" },inplace = True)
print(df.head()) 


Renaming only some of the column names:
   Year WHO_region                             Country Beverage Types  \
0  2023     Africa                        South Africa           Wine   
1  2024     Africa                        South Africa           Wine   
2  2023   Americas  Venezuela (Bolivarian Republic of)           Wine   
3  2024   Americas  Venezuela (Bolivarian Republic of)           Wine   
4  2023     Africa                             Liberia           Wine   

   Display_Value  
0           1.33  
1           1.27  
2           0.04  
3           0.04  
4           0.01  


In [27]:
print(df.loc[:,df.all()])

       Year       WHO_region                                Country  \
0      2023           Africa                           South Africa   
1      2024           Africa                           South Africa   
2      2023         Americas     Venezuela (Bolivarian Republic of)   
3      2024         Americas     Venezuela (Bolivarian Republic of)   
4      2023           Africa                                Liberia   
...     ...              ...                                    ...   
41516  1975  South-East Asia  Democratic People's Republic of Korea   
41517  2017           Europe                                 Serbia   
41518  2017           Europe                               Slovakia   
41519  1976         Americas       Bolivia (Plurinational State of)   
41520  1998         Americas                               Suriname   

      Beverage Types  
0               Wine  
1               Wine  
2               Wine  
3               Wine  
4               Wine  
...      

In [28]:
print("\nFind which years have any non-zero values:")
print(df.loc[:,df.any()])


Find which years have any non-zero values:
       Year       WHO_region                                Country  \
0      2023           Africa                           South Africa   
1      2024           Africa                           South Africa   
2      2023         Americas     Venezuela (Bolivarian Republic of)   
3      2024         Americas     Venezuela (Bolivarian Republic of)   
4      2023           Africa                                Liberia   
...     ...              ...                                    ...   
41516  1975  South-East Asia  Democratic People's Republic of Korea   
41517  2017           Europe                                 Serbia   
41518  2017           Europe                               Slovakia   
41519  1976         Americas       Bolivia (Plurinational State of)   
41520  1998         Americas                               Suriname   

      Beverage Types  Display_Value  
0               Wine           1.33  
1               Wine       

In [29]:
print(df.loc[:,df.notnull().all()])
print("\nRows and columns has a NaN:")
print(df.loc[:,df.isnull().any()])
print("\nDrop rows with any NaNs:")
print(df.dropna(how='any'))

       Year       WHO_region                                Country  \
0      2023           Africa                           South Africa   
1      2024           Africa                           South Africa   
2      2023         Americas     Venezuela (Bolivarian Republic of)   
3      2024         Americas     Venezuela (Bolivarian Republic of)   
4      2023           Africa                                Liberia   
...     ...              ...                                    ...   
41516  1975  South-East Asia  Democratic People's Republic of Korea   
41517  2017           Europe                                 Serbia   
41518  2017           Europe                               Slovakia   
41519  1976         Americas       Bolivia (Plurinational State of)   
41520  1998         Americas                               Suriname   

      Beverage Types  
0               Wine  
1               Wine  
2               Wine  
3               Wine  
4               Wine  
...      

In [30]:
print("\nFrom the 'Year' column, access every other column:")
print(df.loc[:,'Year'::2].head(10))
print("\nAlternate solution:")
print(df.iloc[:,0::2].head(10))


From the 'Year' column, access every other column:
   Year                             Country  Display_Value
0  2023                        South Africa           1.33
1  2024                        South Africa           1.27
2  2023  Venezuela (Bolivarian Republic of)           0.04
3  2024  Venezuela (Bolivarian Republic of)           0.04
4  2023                             Liberia           0.01
5  2024                             Liberia           0.01
6  2023                              Serbia           2.71
7  2024                              Serbia           2.75
8  2023                             Germany           3.01
9  2024                             Germany           3.03

Alternate solution:
   Year                             Country  Display_Value
0  2023                        South Africa           1.33
1  2024                        South Africa           1.27
2  2023  Venezuela (Bolivarian Republic of)           0.04
3  2024  Venezuela (Bolivarian Republic of

In [31]:
print("\nStarting from the 2nd row, access every 5th row:")
print(df.iloc[1::5].head(10))


Starting from the 2nd row, access every 5th row:
    Year             WHO_region       Country Beverage Types  Display_Value
1   2024                 Africa  South Africa           Wine           1.27
6   2023                 Europe        Serbia           Wine           2.71
11  2024                 Europe        Norway           Wine           2.30
16  2023  Eastern Mediterranean         Libya           Wine           0.00
21  2024                 Europe        Cyprus           Wine           0.84
26  2023                 Europe      Slovenia           Wine           3.90
31  2024               Americas        Brazil           Wine           0.36
36  2023                 Africa      Cameroon           Wine           0.06
41  2024  Eastern Mediterranean          Oman           Wine           0.02
46  2023               Americas     Argentina           Wine           2.03
